In [259]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

In [260]:
nr_of_seeds = 30 #nr of seeds in the experiments

In [261]:
#Function that returns best params for each pair of (target_column, target_instance) 
# and their average val rmse, rmse, mae, given a set of config columns (depending on method)
# This funcion can take a dataframe or a path to a csv as input.
# the best settings are chosen using the lowest val rmse
def topk_per_d_per_method(data, config_cols, k=5):
    """
    Return up to the top-k configs per group, ranked by avg_rmse.
    Averages are computed across seeds.
    """
    df = pd.read_csv(data, index_col = [0]) if isinstance(data, str) else data.copy()
    df = df.sort_values(by = ['seed'])

    agg = (df.groupby(config_cols, as_index=False)
            .agg(avg_rmse=('rmse','mean'),
                avg_mae=('mae','mean'),
                n_seeds=('seed','nunique')))
    agg = agg.merge(df, how = 'right')
    agg = agg.sort_values(by = 'avg_rmse').head(k)
    topk_df = agg #agg.groupby(['seed'], group_keys=False).head(k)
    topk_d = topk_df[['avg_rmse', 'rmse', 'mae']]
    return topk_d, topk_df

In [262]:
# Define custom matplotlib colors by their index (only to get consistent colors with friedman vizes)
custom_colors = [plt.get_cmap("tab10")(i) for i in [0, 1, 2, 4]]

# Set this as the default Seaborn palette globally
sns.set_palette(custom_colors)

In [263]:
#Obtain best parameters in order based on test rmse
sub_name = 'species' #select subname (folder) here
dat = pd.read_csv(f'results_{sub_name}/ttb_LS.csv', index_col = [0])
dat = dat.reset_index(drop=True) 
dat['method'] = 'TransferTreeBoost'
dat = dat.sort_values(by = ['seed', 'v', 'source_tree_size', 'target_tree_size', 'k', 'm_0'])

best_LSTransferTreeBoost, best_LSTransferTreeBoost_params = topk_per_d_per_method(
    dat,
['v', 'source_tree_size', 'target_tree_size', 'k', 'm_0'], k=nr_of_seeds
)
best_LSTransferTreeBoost['method'] = 'TTB'

data = pd.read_csv(f'results_{sub_name}/xgb.csv')
xgboost_data = data
xgboost_data['method'] = 'XGBoost'

best_XGBoost, best_XGBoost_params = topk_per_d_per_method(
    xgboost_data,
['v', 'target_tree_size'], k=nr_of_seeds
)
best_XGBoost['method'] = 'XGBoost'

data = pd.read_csv(f'results_{sub_name}/xgb_warmstart.csv')
xgboost_data_warmstart = data
xgboost_data_warmstart ['method'] = 'XGBoost Warmstart'

best_XGBoost_warmstart , best_XGBoost_params_warmstart  = topk_per_d_per_method(
    xgboost_data_warmstart ,
['v', 'target_tree_size'], k=nr_of_seeds
)
best_XGBoost_warmstart ['method'] = 'XGBoost Warmstart'

data = pd.read_csv(f'results_{sub_name}/trada.csv')
trada_data = data
trada_data['method'] = 'TrAdaBoost.R2'

data = pd.read_csv(f'results_{sub_name}/xgb_naive.csv')
xgboost_data_pooled = data
xgboost_data_pooled ['method'] = 'XGBoost Pooled'

best_XGBoost_pooled , best_XGBoost_params_pooled  = topk_per_d_per_method(
    xgboost_data_pooled ,
['v', 'target_tree_size'], k=nr_of_seeds
)
best_XGBoost_pooled['method'] = 'XGBoost Pooled'

best_trada, best_trada_params = topk_per_d_per_method(
    trada_data,
['lr', 'n_estimators', 'tree_size'], k=nr_of_seeds
)
best_trada['method'] = 'TrAdaBoost.R2'

C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_3588\3772309036.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  best_LSTransferTreeBoost['method'] = 'TTB'
C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_3588\3772309036.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  best_XGBoost['method'] = 'XGBoost'
C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_3588\3772309036.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

In [264]:
#Print results
import numpy as np
print(
    np.round(best_LSTransferTreeBoost_params['avg_rmse'].iloc[0], 3), "&",
    np.round(np.std(best_LSTransferTreeBoost_params['rmse']), 3), "&",
    np.round(best_XGBoost_params['avg_rmse'].iloc[0], 3), "&",
    np.round(np.std(best_XGBoost_params['rmse']), 3), "&",
    np.round(best_XGBoost_params_warmstart['avg_rmse'].iloc[0], 3), "&",
    np.round(np.std(best_XGBoost_params_warmstart['rmse']), 3), "&",
    np.round(best_XGBoost_params_pooled['avg_rmse'].iloc[0], 3), "&",
    np.round(np.std(best_XGBoost_params_pooled['rmse']), 3), "&",
    np.round(best_trada_params['avg_rmse'].iloc[0], 3),   "&",
    np.round(np.std(best_trada_params['rmse']), 3)
)

0.106 & 0.022 & 0.119 & 0.024 & 0.113 & 0.03 & 0.105 & 0.026 & 0.1 & 0.021


In [265]:
#Print best hyperparams for TransferTreeBoost
print(float(best_LSTransferTreeBoost_params.iloc[0]['v']), "&",
      float(best_LSTransferTreeBoost_params.iloc[0]['target_tree_size']), "&",
      float(best_LSTransferTreeBoost_params.iloc[0]['source_tree_size']), "&",
      float(best_LSTransferTreeBoost_params.iloc[0]['m_0']), "&",
      float(best_LSTransferTreeBoost_params.iloc[0]['k']))

0.2 & 2.0 & 1.0 & 0.5 & 0.0


In [266]:
data = pd.read_csv(f'results_{sub_name}/ResNet_pooled2.csv')
MLP_data = data

best_pooled, best_pooled_params = topk_per_d_per_method(
    MLP_data,
['learning_rate', 'dropout', 'd_main', 'num_blocks'], k=nr_of_seeds
)

data = pd.read_csv(f'results_{sub_name}/ResNet_finetuned3.csv')
MLP_data = data

best_finetuned, best_finetuned_params = topk_per_d_per_method(
    MLP_data,
['learning_rate', 'dropout', 'd_main', 'num_blocks'], k=nr_of_seeds
)

print(np.round(best_pooled_params['avg_rmse'].iloc[0], 3), "&",
    np.round(np.std(best_pooled_params['rmse']), 3), "&",
     np.round(best_finetuned_params['avg_rmse'].iloc[0], 3), "&",
    np.round(np.std(best_finetuned_params['rmse']), 3), "&",
    np.round(best_XGBoost_params['avg_rmse'].iloc[0], 3), "&",
    np.round(np.std(best_XGBoost_params['rmse']), 3),
    )

0.134 & 0.03 & 0.228 & 0.099 & 0.119 & 0.024


In [267]:
print(
    f"{np.round(best_pooled_params['avg_rmse'].iloc[0], 3)} "
    f"({np.round(np.std(best_pooled_params['rmse']), 3)}) & "
    f"{np.round(best_finetuned_params['avg_rmse'].iloc[0], 3)} "
    f"({np.round(np.std(best_finetuned_params['rmse']), 3)}) & "
    f"{np.round(best_XGBoost_params['avg_rmse'].iloc[0], 3)} "
    f"({np.round(np.std(best_XGBoost_params['rmse']), 3)})"
)

0.134 (0.03) & 0.228 (0.099) & 0.119 (0.024)


In [268]:
best_finetuned

,avg_rmse,rmse,mae
3,0.228492,0.131314,0.089489
8,0.228492,0.117047,0.094455
24,0.228492,0.156352,0.137417
19,0.228492,0.145321,0.123012
48,0.228492,0.149789,0.126004
61,0.228492,0.252728,0.234005
43,0.228492,0.399569,0.331246
38,0.228492,0.297841,0.239236
98,0.228492,0.129840,0.088986
109,0.228492,0.504819,0.485088
